# MuonClip angular/radial RG diagnostics

This notebook tests whether MuonClip is learning mostly in the **angular sector** of a weight matrix while leaving the ordinary singular-value ESD close to its random-matrix baseline.

We work with one named transformer matrix, defaulting to `L00_W_Q`, and compare the initial checkpoint against the best/final/latest checkpoint from the current MuonClip run.

The matrix decomposition is

$$
W_t = U_t \Sigma_t V_t^\top .
$$

The ordinary WeightWatcher ESD sees only

$$
\lambda_i(t)=\sigma_i(t)^2,
$$

so it is a **radial** diagnostic. It is blind to the singular-vector geometry in \(U_t,V_t\).

The angular or quotient object is the polar factor

$$
Q_t = U_t V_t^\top ,
$$

which is obtained by replacing all singular values with one:

$$
\Sigma_t \mapsto I .
$$

For a square Gaussian random matrix, \(Q\) is Haar-distributed on the orthogonal group. For rectangular matrices, the same construction gives a partial isometry on a Stiefel manifold.

## What this notebook tests

We test four related hypotheses.

1. **Radial MP/WeightWatcher hypothesis**

   $$
   \rho_0(\lambda)\approx \rho_{\mathrm{MP}}(\lambda).
   $$

2. **MuonClip radial preservation hypothesis**

   $$
   \rho_t(\lambda)\approx \rho_0(\lambda)
   $$

   even while validation accuracy improves.

3. **Angular learning hypothesis**

   Learning may appear mainly as rotation/concentration of singular vectors:

   $$
   U_0^\top U_t,\qquad V_0^\top V_t .
   $$

4. **Angular quotient RG hypothesis**

   After removing the radial sector via

   $$
   Q_t=U_tV_t^\top,
   $$

   we study angular flow using

   $$
   R_t = Q_0^\top Q_t,\qquad \Delta Q_t=Q_t-Q_0.
   $$

   Ordinary WeightWatcher alpha is **not** defined on \(Q_t\), because every singular value of \(Q_t\) is one:

   $$
   Q_t^\top Q_t=I,\qquad \rho_Q(\lambda)=\delta(\lambda-1).
   $$

   Therefore this notebook reports alpha for the radial sector and separate angular statistics for the quotient sector.

In [ ]:
from __future__ import annotations

from pathlib import Path
import glob
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import weightwatcher as ww
from IPython.display import Image, display, Markdown

from rg_nanogpt_one_head.model import GPT, GPTConfig, transformer_matrix_items

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

print("torch:", torch.__version__)
print("weightwatcher:", getattr(ww, "__version__", "unknown"))

## Configuration

Set `RUN_DIR` to the run you want to analyze.

The notebook tries, in order:

1. `RUN_DIR` from the environment;
2. `RUNROOT/results/muon_clip/seed_<SEED>`;
3. the newest local MuonClip run matching `/tmp/rg-nanogpt-muonclip-*/results/muon_clip/seed_*` or `/private/tmp/...`.

The ordinary long-run trainer should have an initialization checkpoint in

```text
epoch_checkpoints/model_epoch_00p000_step_0000000.pt
```

and later checkpoints such as

```text
checkpoint_best.pt
checkpoint_latest.pt
checkpoint_final.pt
```

In [ ]:
MATRIX_NAME = os.environ.get("RG_MATRIX_NAME", "L00_W_Q")

# Optional manual override:
# RUN_DIR = Path("/tmp/rg-nanogpt-muonclip-3ep-seed4242-.../results/muon_clip/seed_4242")
RUN_DIR = None

# For a running experiment, use "latest"; for a completed comparison, use "best" or "final".
ENDPOINT_PREFERENCE = os.environ.get("RG_ENDPOINT", "best")

print("MATRIX_NAME:", MATRIX_NAME)
print("ENDPOINT_PREFERENCE:", ENDPOINT_PREFERENCE)

In [ ]:
def _mtime(path: Path) -> float:
    try:
        return path.stat().st_mtime
    except FileNotFoundError:
        return -1.0

def discover_run_dir() -> Path:
    if RUN_DIR is not None:
        p = Path(RUN_DIR).expanduser()
        if p.is_dir():
            return p
        raise FileNotFoundError(f"RUN_DIR override does not exist: {p}")

    env_run_dir = os.environ.get("RUN_DIR")
    if env_run_dir:
        p = Path(env_run_dir).expanduser()
        if p.is_dir():
            return p
        warnings.warn(f"RUN_DIR env var set but missing: {p}")

    env_runroot = os.environ.get("RUNROOT")
    if env_runroot:
        root = Path(env_runroot).expanduser()
        candidates = sorted(root.glob("results/muon_clip/seed_*"), key=_mtime)
        if candidates:
            return candidates[-1]

    patterns = [
        "/tmp/rg-nanogpt-muonclip-*/results/muon_clip/seed_*",
        "/private/tmp/rg-nanogpt-muonclip-*/results/muon_clip/seed_*",
        "/tmp/rg-nanogpt-long-muonclip-*/results/muon_clip/seed_*",
        "/private/tmp/rg-nanogpt-long-muonclip-*/results/muon_clip/seed_*",
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(Path(p) for p in glob.glob(pattern))
    candidates = [p for p in candidates if p.is_dir()]
    if not candidates:
        raise FileNotFoundError("Could not discover a MuonClip RUN_DIR. Set RUN_DIR manually.")

    return sorted(set(candidates), key=_mtime)[-1]

RUN_DIR = discover_run_dir()

print("RUN_DIR:")
print(RUN_DIR)
print("\nFiles:")
for p in sorted(RUN_DIR.glob("*")):
    print(" ", p.name)

## Checkpoint discovery

We need two states:

$$
W_0 \quad \text{and} \quad W_T .
$$

The initial state is the model at step zero. The trained state is selected from `checkpoint_best.pt`, `checkpoint_final.pt`, or `checkpoint_latest.pt`.

The notebook also gathers intermediate checkpoints so we can draw an RG-like trajectory:

$$
W_0,W_1,\ldots,W_T .
$$

In [ ]:
def checkpoint_step(path: Path) -> int:
    name = path.name
    match = re.search(r"step[_-](\d+)", name)
    if match:
        return int(match.group(1))
    if name == "checkpoint_best.pt":
        return 10**12 - 3
    if name == "checkpoint_latest.pt":
        return 10**12 - 2
    if name == "checkpoint_final.pt":
        return 10**12 - 1
    return 10**12

def find_initial_checkpoint(run_dir: Path) -> Path:
    candidates = []
    candidates.extend(run_dir.glob("epoch_checkpoints/*step_0000000.pt"))
    candidates.extend(run_dir.glob("epoch_checkpoints/*step_0000000*.pt"))
    candidates.extend(run_dir.glob("model_checkpoints/model_step_0000000.pt"))
    candidates.extend(run_dir.glob("checkpoint_init.pt"))
    candidates = [p for p in candidates if p.is_file()]
    if not candidates:
        raise FileNotFoundError("No step-zero checkpoint found.")
    return sorted(candidates, key=lambda p: (len(str(p)), str(p)))[0]

def find_endpoint_checkpoint(run_dir: Path, preference: str = "best") -> Path:
    order = {
        "best": ["checkpoint_best.pt", "checkpoint_final.pt", "checkpoint_latest.pt"],
        "final": ["checkpoint_final.pt", "checkpoint_best.pt", "checkpoint_latest.pt"],
        "latest": ["checkpoint_latest.pt", "checkpoint_final.pt", "checkpoint_best.pt"],
    }.get(preference, ["checkpoint_best.pt", "checkpoint_final.pt", "checkpoint_latest.pt"])

    for name in order:
        p = run_dir / name
        if p.is_file():
            return p

    candidates = sorted(run_dir.glob("epoch_checkpoints/*.pt"), key=checkpoint_step)
    if candidates:
        return candidates[-1]
    raise FileNotFoundError(f"No endpoint checkpoint found in {run_dir}")

def list_flow_checkpoints(run_dir: Path, endpoint: Path, max_count: int = 12) -> list[Path]:
    candidates = []
    candidates.extend(run_dir.glob("epoch_checkpoints/*.pt"))
    for name in ["checkpoint_best.pt", "checkpoint_latest.pt", "checkpoint_final.pt"]:
        p = run_dir / name
        if p.is_file():
            candidates.append(p)

    unique = []
    seen = set()
    for p in candidates:
        key = str(p)
        if key not in seen and p.is_file():
            seen.add(key)
            unique.append(p)

    unique = sorted(unique, key=lambda p: (checkpoint_step(p), _mtime(p), str(p)))
    if endpoint not in unique and endpoint.is_file():
        unique.append(endpoint)

    if len(unique) <= max_count:
        return unique

    idx = np.linspace(0, len(unique) - 1, max_count).round().astype(int)
    idx = sorted(set(int(i) for i in idx))
    return [unique[i] for i in idx]

INIT_CKPT = find_initial_checkpoint(RUN_DIR)
END_CKPT = find_endpoint_checkpoint(RUN_DIR, ENDPOINT_PREFERENCE)
FLOW_CKPTS = list_flow_checkpoints(RUN_DIR, END_CKPT, max_count=12)

print("Initial checkpoint:")
print(INIT_CKPT)
print("\nEndpoint checkpoint:")
print(END_CKPT)
print("\nFlow checkpoints:")
for p in FLOW_CKPTS:
    print(" ", p)

## Loading checkpoints robustly

The baseline has used a few checkpoint schemas over time. The loader below accepts:

- `model_config` + `model`
- full experiment `config["model"]` + `model`
- `model_args` + `model_state_dict`

It reconstructs the actual `GPT` object and extracts the named matrix through the baseline's own `transformer_matrix_items()` mapping.

In [ ]:
def load_gpt_checkpoint(path: str | Path) -> tuple[GPT, dict]:
    path = Path(path)
    payload = torch.load(path, map_location="cpu", weights_only=False)

    if payload.get("purpose") == "muonclip_walk_full_model_checkpoint":
        cfg = GPTConfig(**payload["model_config"])
        state = payload["model"]
    else:
        if "model_config" in payload:
            model_cfg = payload["model_config"]
        elif "config" in payload:
            config = payload["config"]
            if "model" not in config:
                raise KeyError(f"payload['config'] has no model section: keys={list(config.keys())}")
            model_cfg = config["model"]
        elif "model_args" in payload:
            model_cfg = payload["model_args"]
        else:
            raise KeyError(f"Cannot find model config in {path}. Keys={list(payload.keys())}")

        cfg = model_cfg if isinstance(model_cfg, GPTConfig) else GPTConfig(**model_cfg)

        if "model" in payload:
            state = payload["model"]
        elif "model_state_dict" in payload:
            state = payload["model_state_dict"]
        elif "state_dict" in payload:
            state = payload["state_dict"]
        else:
            raise KeyError(f"Cannot find model state in {path}. Keys={list(payload.keys())}")

    model = GPT(cfg)
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing or unexpected:
        raise RuntimeError(f"State dict mismatch for {path}: missing={missing}, unexpected={unexpected}")
    model.eval()
    return model, payload

def matrix_from_model(model: GPT, matrix_name: str = MATRIX_NAME) -> np.ndarray:
    matrices = {
        name: parameter.detach().float().cpu().numpy()
        for name, _, _, parameter in transformer_matrix_items(model)
    }
    if matrix_name not in matrices:
        raise KeyError(f"{matrix_name} not found. Available: {list(matrices.keys())}")
    return np.asarray(matrices[matrix_name], dtype=float)

def load_matrix_from_checkpoint(path: Path, matrix_name: str = MATRIX_NAME):
    model, payload = load_gpt_checkpoint(path)
    W = matrix_from_model(model, matrix_name)
    return W, model, payload

W0, model0, payload0 = load_matrix_from_checkpoint(INIT_CKPT)
WT, modelT, payloadT = load_matrix_from_checkpoint(END_CKPT)

print("Loaded matrix:", MATRIX_NAME)
print("W0 shape:", W0.shape, "mean/std:", W0.mean(), W0.std())
print("WT shape:", WT.shape, "mean/std:", WT.mean(), WT.std())
assert W0.shape == WT.shape

## SVD, radial sector, and angular quotient

For each matrix:

$$
W=U\Sigma V^\top .
$$

The **radial sector** is the singular-value spectrum:

$$
\Sigma \quad \Longleftrightarrow \quad \{\lambda_i=\sigma_i^2\}.
$$

The **uniform-radial angular quotient** is the polar factor:

$$
Q = U V^\top .
$$

For square \(W\), \(Q\in O(N)\). The quotient removes all singular values:

$$
Q^\top Q = I.
$$

Therefore, the ESD of \(Q\) itself is trivial. The useful angular observables are instead relative rotations and angular displacements:

$$
R_T=Q_0^\top Q_T,\qquad \Delta Q=Q_T-Q_0.
$$

In [ ]:
def svd_parts(W: np.ndarray):
    U, s, Vh = np.linalg.svd(np.asarray(W, dtype=float), full_matrices=False)
    return U, s, Vh

def esd_from_singular_values(s: np.ndarray) -> np.ndarray:
    return np.sort(np.asarray(s, dtype=float) ** 2)

def fro_inner(A, B) -> float:
    return float(np.sum(np.asarray(A) * np.asarray(B)))

def radial_angular_update(Wa: np.ndarray, Wb: np.ndarray) -> dict:
    dW = Wb - Wa
    denom = fro_inner(Wa, Wa)
    coeff = fro_inner(Wa, dW) / denom
    dW_radial = coeff * Wa
    dW_angular = dW - dW_radial
    nd = np.linalg.norm(dW)
    return {
        "coeff": coeff,
        "norm_Wa": np.linalg.norm(Wa),
        "norm_Wb": np.linalg.norm(Wb),
        "norm_dW": nd,
        "norm_radial": np.linalg.norm(dW_radial),
        "norm_angular": np.linalg.norm(dW_angular),
        "radial_fraction": np.linalg.norm(dW_radial) / nd if nd > 0 else np.nan,
        "angular_fraction": np.linalg.norm(dW_angular) / nd if nd > 0 else np.nan,
        "cosine_Wa_Wb": fro_inner(Wa, Wb) / (np.linalg.norm(Wa) * np.linalg.norm(Wb)),
    }

U0, s0, V0h = svd_parts(W0)
UT, sT, VTh = svd_parts(WT)

E0 = esd_from_singular_values(s0)
ET = esd_from_singular_values(sT)

Q0 = U0 @ V0h
QT = UT @ VTh

update_stats = radial_angular_update(W0, WT)
display(pd.Series(update_stats))

## WeightWatcher-style radial ESD plot with MP and Gaussian controls

We compare:

1. the initial matrix \(W_0\);
2. the trained endpoint \(W_T\);
3. a matched Gaussian matrix \(G\) with the same shape and entry standard deviation as \(W_0\);
4. the Marchenko--Pastur density for that shape and variance.

The MP edge for unnormalized eigenvalues \(\lambda=s^2\) is

$$
\lambda_\pm
=
L\sigma^2(1\pm\sqrt{\beta})^2,
\qquad
L=\max(N,M),
\qquad
\beta=\frac{\min(N,M)}{\max(N,M)}.
$$

In [ ]:
rng = np.random.default_rng(12345)
G = rng.normal(0.0, W0.std(), size=W0.shape)
UG, sG, VGh = svd_parts(G)
EG = esd_from_singular_values(sG)

def mp_params(W: np.ndarray, sigma: float | None = None):
    N, M = W.shape
    L = max(N, M)
    S = min(N, M)
    beta = S / L
    if sigma is None:
        sigma = float(W.std())
    lam_minus = L * sigma**2 * (1 - np.sqrt(beta))**2
    lam_plus = L * sigma**2 * (1 + np.sqrt(beta))**2
    return N, M, L, beta, sigma, lam_minus, lam_plus

def mp_density(x, *, L, beta, sigma, lam_minus, lam_plus):
    x = np.asarray(x, dtype=float)
    rho = np.zeros_like(x)
    mask = (x > lam_minus) & (x < lam_plus) & (x > 0)
    rho[mask] = (
        np.sqrt((lam_plus - x[mask]) * (x[mask] - lam_minus))
        / (2 * np.pi * beta * L * sigma**2 * x[mask])
    )
    return rho

def one_matrix_model(W: np.ndarray, name: str = MATRIX_NAME) -> nn.Module:
    class OneMatrixModel(nn.Module):
        def __init__(self, W):
            super().__init__()
            Wt = torch.as_tensor(W, dtype=torch.float32)
            layer = nn.Linear(Wt.shape[1], Wt.shape[0], bias=False)
            with torch.no_grad():
                layer.weight.copy_(Wt)
            self.add_module(name, layer)
    return OneMatrixModel(W)

def ww_details_for_matrix(W: np.ndarray, label: str, plot=False, savedir=None) -> pd.DataFrame:
    watcher = ww.WeightWatcher(model=one_matrix_model(W))
    kwargs = dict(plot=plot, min_evals=20, randomize=True, ERG=False)
    if savedir is not None:
        Path(savedir).mkdir(parents=True, exist_ok=True)
        kwargs["savefig"] = str(savedir)
    details = watcher.analyze(**kwargs)
    details = details.copy()
    details.insert(0, "label", label)
    return details

details_init = ww_details_for_matrix(W0, "initial")
details_end = ww_details_for_matrix(WT, "endpoint")
details_gauss = ww_details_for_matrix(G, "matched_gaussian")

display(pd.concat([details_init, details_end, details_gauss], ignore_index=True))

In [ ]:
def plot_weightwatcher_style_esd(spectra: dict[str, np.ndarray], W_reference: np.ndarray, title: str, nbins: int = 18):
    all_evals = np.concatenate([v[np.isfinite(v) & (v > 0)] for v in spectra.values()])
    N, M, L, beta, sigma, lam_minus, lam_plus = mp_params(W_reference)
    lo = max(all_evals.min(), 1e-12)
    hi = max(all_evals.max(), lam_plus) * 1.15
    edges = np.geomspace(lo, hi, nbins + 1)
    centers = np.sqrt(edges[:-1] * edges[1:])
    widths = np.diff(edges)

    x_mp = np.geomspace(max(lo, lam_minus, 1e-12), max(lam_plus, lo * 1.01), 1500)
    rho_mp = mp_density(x_mp, L=L, beta=beta, sigma=sigma, lam_minus=lam_minus, lam_plus=lam_plus)

    fig, ax = plt.subplots(figsize=(11, 7.5))

    labels = list(spectra.keys())
    first = labels[0]
    rho_first, _ = np.histogram(spectra[first], bins=edges, density=True)
    m = rho_first > 0
    ax.bar(centers[m], rho_first[m], width=widths[m] * 0.82, align="center", alpha=0.95, label=first)

    markers = ["s", "^", "D", "v"]
    for i, label in enumerate(labels[1:]):
        rho, _ = np.histogram(spectra[label], bins=edges, density=True)
        m = rho > 0
        ax.plot(centers[m], rho[m], marker=markers[i % len(markers)], linestyle="--" if "Gaussian" in label or "gaussian" in label else "-", linewidth=2.2, markersize=5, label=label)

    mp_mask = rho_mp > 0
    ax.plot(x_mp[mp_mask], rho_mp[mp_mask], linewidth=3, label="Marchenko–Pastur")
    ax.axvline(lam_plus, linestyle=":", linewidth=2, label=fr"MP $\lambda_+={lam_plus:.4f}$")

    try:
        row = details_end.iloc[0]
        alpha = float(row.get("alpha", np.nan))
        xmin = float(row.get("xmin", np.nan))
        D = float(row.get("D", np.nan))
        if np.isfinite(xmin) and xmin > 0:
            ax.axvline(xmin, linewidth=2.0, label=fr"WW $\lambda_{{min}}={xmin:.3g}$")
        suffix = fr"$\alpha_T={alpha:.3f};\ D_{{KS}}={D:.3f}$"
    except Exception:
        suffix = ""

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"Eigenvalue $\lambda=s^2$")
    ax.set_ylabel(r"Spectral density $\rho(\lambda)$")
    ax.set_title(title + ("\n" + suffix if suffix else ""), fontsize=16)
    ax.legend(fontsize=10)
    ax.grid(True, which="both", alpha=0.22)
    plt.tight_layout()
    return fig, ax

plot_weightwatcher_style_esd(
    {"initial W0": E0, "endpoint WT": ET, "matched Gaussian": EG},
    W0,
    f"Log-Log ESD for {MATRIX_NAME}: initial vs endpoint",
    nbins=18,
)
plt.show()

## Native WeightWatcher plots

The custom plot above is for clean comparisons and stable binning. We also generate the **native WeightWatcher plots** using exactly:

```python
watcher.analyze(plot=True, savefig=savedir, min_evals=20, randomize=True, ERG=False)
```

The native log-log ESD file is generated by WeightWatcher as:

```text
ww.layer<layer_id>.esd.png
```

In [ ]:
native_root = Path("/tmp/rg_muonclip_angular_radial_native_ww")
native_root.mkdir(parents=True, exist_ok=True)

native_init_dir = native_root / "initial"
native_end_dir = native_root / "endpoint"

native_init = ww_details_for_matrix(W0, "initial", plot=True, savedir=native_init_dir)
native_end = ww_details_for_matrix(WT, "endpoint", plot=True, savedir=native_end_dir)

for label, d, details in [("initial", native_init_dir, native_init), ("endpoint", native_end_dir, native_end)]:
    layer_id = int(details.iloc[0]["layer_id"])
    esd_path = d / f"ww.layer{layer_id}.esd.png"
    print(label, "layer_id=", layer_id, "path=", esd_path, "exists=", esd_path.is_file())
    if esd_path.is_file():
        display(Markdown(f"### Native WeightWatcher ESD: {label}"))
        display(Image(filename=str(esd_path)))

## Angular quotient diagnostics

Now we remove singular values completely:

$$
Q_0=U_0V_0^\top,\qquad Q_T=U_TV_T^\top .
$$

The relative rotation is

$$
R_T=Q_0^\top Q_T .
$$

For square matrices, \(R_T\in O(N)\). Its eigenvalues lie on the unit circle:

$$
z_j=e^{i\theta_j}.
$$

We compare \(R_T\) to a Haar-random orthogonal matrix, and we inspect the chordal angular displacement:

$$
\Delta Q=Q_T-Q_0.
$$

In [ ]:
def haar_orthogonal(n: int, rng=None) -> np.ndarray:
    if rng is None:
        rng = np.random.default_rng()
    A = rng.normal(size=(n, n))
    Q, R = np.linalg.qr(A)
    signs = np.sign(np.diag(R))
    signs[signs == 0] = 1
    Q = Q * signs
    return Q

def eigenangles_of_orthogonal(R: np.ndarray) -> np.ndarray:
    z = np.linalg.eigvals(R)
    theta = np.abs(np.angle(z))
    theta = np.minimum(theta, 2*np.pi - theta)
    return np.sort(theta)

def ipr_columns(A: np.ndarray) -> np.ndarray:
    A = np.asarray(A, dtype=float)
    norms = np.linalg.norm(A, axis=0, keepdims=True)
    norms[norms == 0] = 1
    X = A / norms
    return np.sum(X**4, axis=0)

R_emp = Q0.T @ QT
theta_emp = eigenangles_of_orthogonal(R_emp)

Q_haar = haar_orthogonal(Q0.shape[0], rng)
theta_haar = eigenangles_of_orthogonal(Q_haar)

left_overlap = np.abs(U0.T @ UT)**2
right_overlap = np.abs(V0h @ VTh.T)**2

angular_stats = {
    "polar_chordal_norm": np.linalg.norm(QT - Q0, "fro"),
    "polar_chordal_norm_per_dim": np.linalg.norm(QT - Q0, "fro") / np.sqrt(QT.shape[0]),
    "mean_abs_angle": float(np.mean(theta_emp)),
    "median_abs_angle": float(np.median(theta_emp)),
    "max_abs_angle": float(np.max(theta_emp)),
    "left_diag_overlap_mean": float(np.mean(np.diag(left_overlap))),
    "right_diag_overlap_mean": float(np.mean(np.diag(right_overlap))),
    "U_ipr_mean_initial": float(np.mean(ipr_columns(U0))),
    "U_ipr_mean_endpoint": float(np.mean(ipr_columns(UT))),
    "V_ipr_mean_initial": float(np.mean(ipr_columns(V0h.T))),
    "V_ipr_mean_endpoint": float(np.mean(ipr_columns(VTh.T))),
    "haar_ipr_reference_left": 3.0 / (U0.shape[0] + 2.0),
    "haar_ipr_reference_right": 3.0 / (V0h.shape[1] + 2.0),
}
display(pd.Series(angular_stats))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
bins = np.linspace(0, np.pi, 25)
ax.hist(theta_emp, bins=bins, density=True, histtype="step", linewidth=2.5, label="relative rotation Q0ᵀQT")
ax.hist(theta_haar, bins=bins, density=True, histtype="step", linewidth=2.5, linestyle="--", label="Haar orthogonal null")
ax.set_xlabel(r"Eigenangle $|\theta|$")
ax.set_ylabel("density")
ax.set_title(f"Angular quotient eigenangle distribution for {MATRIX_NAME}")
ax.legend()
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
im0 = axes[0].imshow(left_overlap, aspect="auto", origin="lower")
axes[0].set_title(r"$|U_0^\top U_T|^2$")
axes[0].set_xlabel("trained singular vector")
axes[0].set_ylabel("initial singular vector")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(right_overlap, aspect="auto", origin="lower")
axes[1].set_title(r"$|V_0^\top V_T|^2$")
axes[1].set_xlabel("trained singular vector")
axes[1].set_ylabel("initial singular vector")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

plt.tight_layout()
plt.show()

## Radial-preserving and angular-preserving nulls

Given

$$
W_T=U_T\Sigma_TV_T^\top,
$$

we build two diagnostic nulls.

### 1. Radial-preserving, angular-randomized null

Keep the trained singular values but replace angular information with Haar-random factors:

$$
W_{\mathrm{radial\ only}}
=
U_{\mathrm{Haar}}\Sigma_TV_{\mathrm{Haar}}^\top .
$$

This matrix has exactly the same ESD and alpha as \(W_T\), but all learned singular-vector structure has been erased.

### 2. Angular-preserving, radial-uniformized quotient

Keep the trained angular factor but set all singular values equal to one:

$$
Q_T=U_TV_T^\top .
$$

This matrix has a trivial ESD, so we analyze angular displacement and localization rather than ordinary alpha.

In [ ]:
def radial_preserving_haar_null(s: np.ndarray, shape: tuple[int, int], rng=None) -> np.ndarray:
    if rng is None:
        rng = np.random.default_rng()
    N, M = shape
    k = min(N, M)
    Uh = haar_orthogonal(N, rng)[:, :k]
    Vh = haar_orthogonal(M, rng)[:, :k]
    return Uh @ np.diag(s[:k]) @ Vh.T

W_radial_only = radial_preserving_haar_null(sT, WT.shape, rng)
s_radial_only = np.linalg.svd(W_radial_only, compute_uv=False)
E_radial_only = esd_from_singular_values(s_radial_only)

print("Max ESD difference between WT and radial-only Haar null:")
print(np.max(np.abs(np.sort(ET) - np.sort(E_radial_only))))

radial_only_details = ww_details_for_matrix(W_radial_only, "radial_only_haar")
display(pd.concat([details_end, radial_only_details], ignore_index=True))

plot_weightwatcher_style_esd(
    {"endpoint WT": ET, "radial-only Haar null": E_radial_only, "matched Gaussian": EG},
    W0,
    f"Radial-preserving angular randomization for {MATRIX_NAME}",
    nbins=18,
)
plt.show()

## Flow across available checkpoints

For each checkpoint \(t\), we record:

- radial WeightWatcher alpha;
- WeightWatcher `rand_distance`;
- \(\|W_t\|_F\);
- angular quotient distance \(\|Q_t-Q_0\|_F/\sqrt{N}\);
- mean diagonal singular-vector overlaps;
- mean IPR of singular vectors.

The line

$$
\alpha=2
$$

is drawn as the proposed marginal critical reference. This notebook tests whether the observed radial flow approaches that reference; it does not assume the answer in advance.

In [ ]:
def checkpoint_metadata(path: Path, payload: dict) -> dict:
    out = {"path": str(path), "name": path.name}
    for k in ["step", "epoch", "tokens_seen", "val_acc", "val_loss"]:
        if k in payload:
            out[k] = payload[k]
    if "metrics" in payload and isinstance(payload["metrics"], dict):
        for k in ["val_acc", "val_loss", "train_loss"]:
            if k in payload["metrics"]:
                out[k] = payload["metrics"][k]
    out.setdefault("step", checkpoint_step(path))
    return out

def one_row_for_checkpoint(path: Path, W_ref: np.ndarray, Q_ref: np.ndarray) -> dict:
    W, model, payload = load_matrix_from_checkpoint(path)
    U, s, Vh = svd_parts(W)
    E = esd_from_singular_values(s)
    Q = U @ Vh

    try:
        d = ww_details_for_matrix(W, path.name, plot=False)
        row = d.iloc[0]
        alpha = float(row.get("alpha", np.nan))
        D = float(row.get("D", np.nan))
        rand_distance = float(row.get("rand_distance", np.nan))
    except Exception as e:
        alpha, D, rand_distance = np.nan, np.nan, np.nan
        print("WeightWatcher failed on", path, e)

    Uref, _, Vrefh = svd_parts(W_ref)
    left_overlap = np.abs(Uref.T @ U)**2
    right_overlap = np.abs(Vrefh @ Vh.T)**2

    meta = checkpoint_metadata(path, payload)
    meta.update({
        "alpha": alpha,
        "D": D,
        "rand_distance": rand_distance,
        "frob_norm": float(np.linalg.norm(W)),
        "spectral_max": float(E.max()),
        "spectral_mean": float(E.mean()),
        "polar_chordal_norm_per_dim": float(np.linalg.norm(Q - Q_ref, "fro") / np.sqrt(Q.shape[0])),
        "left_diag_overlap_mean": float(np.mean(np.diag(left_overlap))),
        "right_diag_overlap_mean": float(np.mean(np.diag(right_overlap))),
        "U_ipr_mean": float(np.mean(ipr_columns(U))),
        "V_ipr_mean": float(np.mean(ipr_columns(Vh.T))),
    })
    return meta

flow_rows = []
for p in FLOW_CKPTS:
    try:
        flow_rows.append(one_row_for_checkpoint(p, W0, Q0))
    except Exception as e:
        print("Skipping checkpoint:", p, "error:", repr(e))

flow = pd.DataFrame(flow_rows)
display(flow)

In [ ]:
if len(flow):
    x = flow["step"].to_numpy(dtype=float)
    if np.all(x > 1e11) or len(np.unique(x)) < 2:
        x = np.arange(len(flow), dtype=float)
        xlabel = "checkpoint index"
    else:
        xlabel = "optimizer step"

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0,0].plot(x, flow["alpha"], "o-", linewidth=2)
    axes[0,0].axhline(2.0, linestyle="--", linewidth=2, label=r"$\alpha=2$")
    axes[0,0].set_title("Radial WeightWatcher alpha")
    axes[0,0].set_xlabel(xlabel)
    axes[0,0].set_ylabel(r"$\alpha$")
    axes[0,0].legend()

    axes[0,1].plot(x, flow["rand_distance"], "o-", linewidth=2)
    axes[0,1].set_title("WeightWatcher RAND distance")
    axes[0,1].set_xlabel(xlabel)
    axes[0,1].set_ylabel("rand_distance")

    axes[1,0].plot(x, flow["polar_chordal_norm_per_dim"], "o-", linewidth=2)
    axes[1,0].set_title(r"Angular quotient distance $\|Q_t-Q_0\|_F/\sqrt{N}$")
    axes[1,0].set_xlabel(xlabel)

    axes[1,1].plot(x, flow["left_diag_overlap_mean"], "o-", label="left U", linewidth=2)
    axes[1,1].plot(x, flow["right_diag_overlap_mean"], "s-", label="right V", linewidth=2)
    axes[1,1].set_title("Mean diagonal singular-vector overlap")
    axes[1,1].set_xlabel(xlabel)
    axes[1,1].legend()

    plt.tight_layout()
    plt.show()

## Interpretation checklist

### Case A: radial learning

If

$$
\rho_T(\lambda)
$$

moves away from MP, RAND distance increases, and alpha moves toward the proposed marginal value

$$
\alpha\to 2,
$$

then the ordinary radial WeightWatcher/RG picture is capturing the dominant learning signal.

### Case B: angular learning with radial preservation

If

$$
\rho_T(\lambda)\approx \rho_0(\lambda)
$$

but

$$
\|Q_T-Q_0\|_F
$$

grows, diagonal singular-vector overlap drops, or IPR/localization changes, then MuonClip is learning primarily in the quotient/angular sector.

### Case C: both sectors move

If both radial and angular diagnostics move, then MuonClip has a coupled radial/angular RG flow.

The most important caveat is that ordinary WeightWatcher alpha is a radial observable. Once we replace

$$
\Sigma\mapsto I,
$$

the ordinary ESD is a delta function and alpha is not the correct quotient-space observable. The angular RG must be defined using rotations, principal angles, IPR/localization, or an ESD of an angular displacement such as

$$
\Delta Q_t=Q_t-Q_0.
$$